# Power System Simulation — Final Presentation

Group 5XWG0. Members: Oliwia Hejduk, Joost van der Steen, Jarno de Poorter,
Alejandro Solá Bengoechea.

This notebook walks through everything our package can do, in the order the
three assignments are organized.


## 1. Setup

The package is installable with `uv sync` from the repository root. All the
public APIs are importable from `power_system_simulation`. The small test
network used in this notebook is **not** committed to the repo — it lives
in a local folder. Set `DATA_DIR` below to wherever you have downloaded the
small_network from SharePoint.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

# Adjust this path to wherever the small_network lives on your machine.
# (Each teammate's path differs because the data isn't committed to the repo.)
DATA_DIR = Path.home() / "Documents" / "Q4" / "CBL_PSCS " / "small_network" / "input"

NETWORK_JSON = DATA_DIR / "input_network_data.json"
ACTIVE_PROFILE = DATA_DIR / "active_power_profile.parquet"
REACTIVE_PROFILE = DATA_DIR / "reactive_power_profile.parquet"
EV_PROFILE = DATA_DIR / "ev_active_power_profile.parquet"

# From meta_data.json
FEEDER_IDS = [16, 20]

print("network:", NETWORK_JSON.exists())
print("active :", ACTIVE_PROFILE.exists())
print("reactive:", REACTIVE_PROFILE.exists())
print("ev     :", EV_PROFILE.exists())

## 2. Assignment 1 — Graph processing

`GraphProcessor` is an undirected graph with a designated source vertex,
intended for radial power-grid topology analysis. It exposes two
functionalities used by later assignments:

- `find_downstream_vertices(edge_id)` — every vertex on the far side of an
  edge from the source.
- `find_alternative_edges(disabled_edge_id)` — every currently-disabled
  edge that could be turned on to keep the graph connected if the given
  edge were removed.


In [ ]:
from power_system_simulation.graph_processing import GraphProcessor

# Small toy example matching the docstring drawing:
#
#   0 (source) --edge_1(on)-- 2 --edge_9(on)-- 10
#       |                     |
#       |                edge_7(off)
#       |                     |
#       --------edge_3(on)-- 4
#       |                     |
#       |                edge_8(off)
#       |                     |
#       --------edge_5(on)-- 6
gp = GraphProcessor(
    vertex_ids=[0, 2, 4, 6, 10],
    edge_ids=[1, 3, 5, 7, 8, 9],
    vertex_edge_id_pairs=[(0, 2), (0, 4), (0, 6), (2, 4), (4, 6), (2, 10)],
    edge_enabled=[True, True, True, False, False, True],
    source_vertex_id=0,
)

print("downstream of edge 1:", gp.find_downstream_vertices(1))
print("downstream of edge 3:", gp.find_downstream_vertices(3))
print("alternatives if edge 3 fails:", gp.find_alternative_edges(3))
print("alternatives if edge 9 fails:", gp.find_alternative_edges(9))

## 3. Assignment 2 — Time-series power flow

`PowerFlowRunner` loads a PGM JSON network and a matching pair of active /
reactive load profiles, builds the time-series batch update, runs the
calculation, and aggregates the result into two summary tables.


In [ ]:
from power_system_simulation.pgm_runner import PowerFlowRunner

runner = PowerFlowRunner(
    network_json_path=NETWORK_JSON,
    active_profile_path=ACTIVE_PROFILE,
    reactive_profile_path=REACTIVE_PROFILE,
)
runner.run()
print("ran power flow over", runner.active_profile.shape[0], "timestamps")

### 3.1 Voltage summary — one row per timestamp

In [ ]:
voltage_table = runner.aggregate_power_flow()
voltage_table.head()

### 3.2 Line summary — one row per line

In [ ]:
line_table = runner.node_table()
line_table

### 3.3 A picture — voltage band over time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(voltage_table.index, voltage_table["Maximum voltage (pu)"], label="max p.u.")
ax.plot(voltage_table.index, voltage_table["Minimum voltage (pu)"], label="min p.u.")
ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
ax.set_ylabel("Voltage (p.u.)")
ax.set_title("Network-wide voltage band over the simulation window")
ax.legend()
fig.autofmt_xdate()
plt.show()

## 4. Assignment 3 — LV grid analytics

A3 ties A1 and A2 together. We start with an `LVGrid` that validates the
input data; the three analyses below all build on top of it.


### 4.1 Input validation

`LVGrid` runs ten validation checks on construction (PGM validity, exactly
one transformer + one source, valid feeder IDs, feeders start at the
transformer's LV side, initial state is connected and acyclic, matching
timestamps and IDs across the three profiles, enough EV profiles). If any
check fails it raises a specific exception so the caller knows what went
wrong.


In [ ]:
from power_system_simulation.lv_grid import LVGrid, OptimizationCriteria

grid = LVGrid(
    network_json_path=NETWORK_JSON,
    active_profile_path=ACTIVE_PROFILE,
    reactive_profile_path=REACTIVE_PROFILE,
    ev_profile_path=EV_PROFILE,
    feeder_ids=FEEDER_IDS,
)

print("transformer id    :", grid.transformer_id)
print("source id         :", grid.source_id)
print("LV busbar (node)  :", grid.transformer_lv_node)
print("sym_load ids      :", grid.sym_load_ids)
print("feeder line ids   :", grid.feeder_ids)
print("timestamps        :", grid.active_profile.shape[0])
print("EV profiles in pool:", grid.ev_profile.shape[1])

Quick demonstration that a bad input is rejected — here we pass a
non-existent feeder ID and `FeederIdNotALineError` is raised.


In [ ]:
from power_system_simulation.lv_grid import FeederIdNotALineError

try:
    LVGrid(
        network_json_path=NETWORK_JSON,
        active_profile_path=ACTIVE_PROFILE,
        reactive_profile_path=REACTIVE_PROFILE,
        ev_profile_path=EV_PROFILE,
        feeder_ids=[16, 99999],  # 99999 is not a line
    )
except FeederIdNotALineError as e:
    print("caught as expected:", e)

### 4.2 EV penetration analysis  *(presenter to fill in)*

Given an EV penetration level (e.g. 20%), randomly assign EV charging
profiles from the pool to a subset of households, distributed across the
feeders according to the spec. Then run a time-series power flow with the
new loads and produce the same two aggregation tables as in Assignment 2.

**Demo cells go below — please add them when your implementation lands.**


In [ ]:
# TODO: presenter to fill in.
# Suggested shape:
#
#   from power_system_simulation.ev_penetration import ev_penetration_analysis
#   voltage_table, line_table = ev_penetration_analysis(
#       grid, penetration_level=0.2, random_seed=42,
#   )
#   voltage_table.head()


### 4.3 Optimal tap position 

Sweep every possible tap position of the transformer, re-run the full
time-series power flow for each, and return the tap that minimises the
chosen criterion (total energy loss across lines, or average voltage
deviation from 1 p.u.). The criterion is a user-selectable argument. The default is set as the voltage deviation. 


In [ ]:
optimal_tap_energy = grid.optimize_tap_position(criteria=OptimizationCriteria.MIN_ENERGY_LOSS)
print(f"Optimal tap (min voltage deviation): {optimal_tap_energy}")
optimal_tap_energy = grid.optimize_tap_position(criteria=OptimizationCriteria.MIN_VOLTAGE_DEVIATION)
print(f"Optimal tap (min energy loss): {optimal_tap_energy}")

### 4.4 N-1 calculation

Given a line that is going out of service, find every currently-disabled
line that could be switched in to keep the grid connected, then run the
full time-series power flow for each alternative. Return a summary table
with one row per alternative and the maximum loading information.


In [ ]:
# Simulate line fault on some line in the network
faulted_line_id = 18

# Execute the N-1 contingency calculation from our validated grid object
n1_summary_table = grid.calculate_n_minus_1(line_id=faulted_line_id)

# Render the operational summary table
if n1_summary_table.empty:
    print("No valid alternative configurations exist.")
else:
    print("Scenarios ranked by maximum strain:")

n1_summary_table

## 5. Collaboration and lessons learned  *(~2 minutes, all of us)*

**How we worked together:**

- All work merged through pull requests with branch protection (2 approvals
  required, CI must pass, conversations resolved).
- Branches were stacked when one chunk depended on another that hadn't
  merged yet (e.g. A3 validation stacked on the A2 runner branch).
- We used pre-commit + ruff so formatting was never a CI surprise.
- pytest with 95%+ coverage was enforced by CI.

**On AI tools:**

We used AI assistants heavily across the project. Where they helped:
scaffolding boilerplate code, explaining unfamiliar libraries like
`power-grid-model`, drafting test cases, and catching small bugs during
PR reviews. Where they didn't: we lost time once when an AI suggested a
data format that didn't match the actual parquet files, and it can't
replace a teammate decision about whose API wins when two of us start
the same chunk in parallel. The lesson is the obvious one — AI is great
for unblocking and explaining, but it can't make team decisions for you.

**Things we'd do differently next time:**

- Lock down the API shape together at the start of each assignment so two
  people don't write competing versions of the same thing in parallel.
- Talk about misunderstandings in the input data format earlier (we lost a
  day on a parser that assumed the wrong tuple shape).
- Decide upfront whether each new feature gets stacked on an open PR or
  branched off main, instead of figuring it out per-PR.
- Use the mentor and TA for design questions earlier rather than during
  the final week.
